In [8]:
# Instalação e Imports
!pip install -q torch_snippets
from torch_snippets import *
from torchvision.models import vgg19, VGG19_Weights
import torch.nn as nn
import torch.optim as optim

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [9]:
# Configuração das Imagens
content_path = 'content.png'
style_path = 'style.jpg'

In [10]:
# Baixa as imagens
!wget -q https://easydrawingguides.com/wp-content/uploads/2016/10/how-to-draw-an-elephant-featured-image-1200.png -O content.png
!wget -q https://i.pinimg.com/736x/e5/44/54/e544542aa0ae414abdb0778799228637.jpg -O style.jpg

In [11]:
#  pipelines de Processamento
preprocess = T.Compose([
    T.Resize(512), # Redimensiona
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.Lambda(lambda x: x.mul_(255))
])

postprocess = T.Compose([
    T.Lambda(lambda x: x.mul_(1./255)),
    T.Normalize(mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225], std=[1/0.229, 1/0.224, 1/0.255])
])

In [13]:
#  funções de Suporte
class GramMatrix(nn.Module):
    def forward(self, input):
        b, c, h, w = input.size()
        feat = input.view(b, c, h*w)
        G = feat @ feat.transpose(1, 2)
        return G.div_(h*w)

class vgg19_modified(nn.Module):
    def __init__(self):
        super().__init__()
        features = list(vgg19(weights=VGG19_Weights.DEFAULT).features)
        self.features = nn.ModuleList(features).eval()
    def forward(self, x, layers=[]):
        res = []
        for ix, layer in enumerate(self.features):
            x = layer(x)
            if ix in layers: res.append(x)
        return res

vgg = vgg19_modified().to(device)

In [15]:
# preparação para o Treino
from PIL import Image # Importa a biblioteca PIL para manipulação de imagens

def load_image(path):
    img_np = read(path, 1) # `read` from torch_snippets returns a numpy array
    img_pil = Image.fromarray(img_np) # Converte o array numpy para PIL Image
    return preprocess(img_pil)[None].to(device)

# carrega as imagens originais
img_content = load_image(content_path)
img_style = load_image(style_path)

# criamos a imagem
opt_img = img_content.data.clone().requires_grad_(True)

content_layers = [18] # camada profunda para estrutura
style_layers = [0, 5, 10, 19, 28] # várias camadas para capturar texturas

target_content = [f.detach() for f in vgg(img_content, content_layers)]
target_style = [GramMatrix()(f).detach() for f in vgg(img_style, style_layers)]

In [ ]:
optimizer = optim.Adam([opt_img], lr=2.0)
mse = nn.MSELoss()

for i in range(500):
    optimizer.zero_grad()

    # Extrai features da imagem que está sendo criada
    out_content = vgg(opt_img, content_layers)
    out_style = vgg(opt_img, style_layers)

    # Cálculo das perdas
    loss_content = sum([mse(o, t) for o, t in zip(out_content, target_content)])
    loss_style = sum([mse(GramMatrix()(o), t) for o, t in zip(out_style, target_style)])

    total_loss = (loss_content * 1) + (loss_style * 1e6)

    total_loss.backward()
    optimizer.step()

    # Garante que os pixels
    opt_img.data.clamp_(-10, 10)

    if (i+1) % 100 == 0:
        print(f"Iteração {i+1} | Loss: {total_loss.item():.2f}")
        with torch.no_grad():
            display_img = postprocess(opt_img[0]).permute(1,2,0).cpu()
            show(display_img)